# Notebook 03 — V3: El doble filo del empleo (heatmap)

**Visualización:** Heatmap en Flourish  
**Pregunta:** ¿Cómo interactúan las horas de trabajo y la presencia de hijos sobre el bienestar, y difiere ese patrón entre hombres y mujeres?  
**Nota codificación:** `chldhhe` en ESS R11 es **binaria**: 1 = tiene hijos en el hogar, 2 = sin hijos  
**Output:** `data/processed/v3_heatmap_empleo.csv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path('../')
OUT  = ROOT / 'data' / 'processed'

print('Setup OK')

Setup OK


## 1. Cargar ESS R11 y revisar variables clave

In [2]:
ess11 = pd.read_csv(OUT / 'ESS11_clean.csv', low_memory=False)
print(f'ESS R11: {len(ess11):,} filas')

for v in ['gndr', 'IDX_WELLBEING', 'wkhtot', 'pdwrk', 'hswrk', 'chldhhe']:
    n = ess11[v].notna().sum()
    print(f'  {v}: {n:,} válidos ({n/len(ess11)*100:.1f}%)')

print()
print('Valores únicos chldhhe (tras limpieza):', sorted(ess11['chldhhe'].dropna().unique()))
print('  → 1 = tiene hijos en hogar, 2 = sin hijos (variable binaria en ESS R11)')

ESS R11: 50,116 filas
  gndr: 50,116 válidos (100.0%)
  IDX_WELLBEING: 50,003 válidos (99.8%)
  wkhtot: 41,923 válidos (83.7%)
  pdwrk: 50,116 válidos (100.0%)
  hswrk: 50,116 válidos (100.0%)
  chldhhe: 33,499 válidos (66.8%)

Valores únicos chldhhe (tras limpieza): [1.0, 2.0]
  → 1 = tiene hijos en hogar, 2 = sin hijos (variable binaria en ESS R11)


## 2. Filtrar población activa

In [3]:
# Personas con trabajo remunerado (pdwrk=1) O amas/amos de casa (hswrk=1)
activos = ess11[
    (ess11['pdwrk'] == 1) | (ess11['hswrk'] == 1)
].copy()

# Amas/amos de casa sin horas reportadas → 0h remuneradas (quintil más bajo)
activos['wkhtot_hm'] = activos['wkhtot'].fillna(0)

# Descartar sin IDX_WELLBEING
activos = activos.dropna(subset=['IDX_WELLBEING'])

print(f'Población activa con IDX_WELLBEING: {len(activos):,}')
print(f'  Hombres: {(activos["gndr"]==1).sum():,}')
print(f'  Mujeres: {(activos["gndr"]==2).sum():,}')
print()
print('wkhtot_hm:', activos['wkhtot_hm'].describe().round(1).to_dict())

Población activa con IDX_WELLBEING: 30,137
  Hombres: 13,794
  Mujeres: 16,343

wkhtot_hm: {'count': 30137.0, 'mean': 36.2, 'std': 19.7, 'min': 0.0, '25%': 30.0, '50%': 40.0, '75%': 44.0, 'max': 168.0}


## 3. Crear quintiles de horas y grupos de hijos

In [4]:
# Quintiles de horas (qcut con duplicates='drop' para evitar errores por valores repetidos)
_, bins = pd.qcut(activos['wkhtot_hm'], q=5, retbins=True, duplicates='drop')
labels_q = [f'Q{i+1} ({int(lo)}-{int(hi)}h)' for i, (lo, hi) in enumerate(zip(bins[:-1], bins[1:]))]

activos['quintil_horas'] = pd.qcut(
    activos['wkhtot_hm'], q=5, labels=labels_q, duplicates='drop'
)

print(f'Quintiles generados: {len(labels_q)} — {labels_q}')
print(activos['quintil_horas'].value_counts().sort_index())

Quintiles generados: 4 — ['Q1 (0-25h)', 'Q2 (25-40h)', 'Q3 (40-45h)', 'Q4 (45-168h)']
quintil_horas
Q1 (0-25h)       6261
Q2 (25-40h)     14752
Q3 (40-45h)      3566
Q4 (45-168h)     5558
Name: count, dtype: int64


In [5]:
# Grupo de hijos: binario según chldhhe (1=con hijos, 2=sin hijos)
activos['grupo_hijos'] = activos['chldhhe'].map({1: 'Con hijos', 2: 'Sin hijos'})

print('Distribución grupo_hijos:')
print(activos['grupo_hijos'].value_counts())
print(f'Sin dato: {activos["grupo_hijos"].isna().sum():,}')

Distribución grupo_hijos:
grupo_hijos
Sin hijos    9431
Con hijos    7158
Name: count, dtype: int64
Sin dato: 13,548


In [6]:
# Filtrar filas con todos los campos necesarios
activos_hm = activos.dropna(subset=['quintil_horas', 'grupo_hijos']).copy()
activos_hm['genero'] = activos_hm['gndr'].map({1: 'Hombres', 2: 'Mujeres'})

n_quintiles = activos_hm['quintil_horas'].nunique()
print(f'Filas válidas para el heatmap: {len(activos_hm):,}')
print(f'  Hombres: {(activos_hm["genero"]=="Hombres").sum():,}')
print(f'  Mujeres: {(activos_hm["genero"]=="Mujeres").sum():,}')
print(f'Celdas esperadas: 2 géneros × {n_quintiles} quintiles × 2 grupos = {2*n_quintiles*2}')

Filas válidas para el heatmap: 16,589
  Hombres: 7,979
  Mujeres: 8,610
Celdas esperadas: 2 géneros × 4 quintiles × 2 grupos = 16


## 4. Calcular IDX_WELLBEING medio por celda

In [7]:
hm = (
    activos_hm
    .groupby(['genero', 'quintil_horas', 'grupo_hijos'], observed=True)['IDX_WELLBEING']
    .agg(idx_wellbeing='mean', n='count')
    .reset_index()
)
hm['idx_wellbeing'] = hm['idx_wellbeing'].round(3)

print(f'Celdas en el heatmap: {len(hm)}')
print()
print(hm.to_string(index=False))

Celdas en el heatmap: 16

 genero quintil_horas grupo_hijos  idx_wellbeing    n
Hombres    Q1 (0-25h)   Con hijos          6.326  472
Hombres    Q1 (0-25h)   Sin hijos          6.469  722
Hombres   Q2 (25-40h)   Con hijos          6.474 1302
Hombres   Q2 (25-40h)   Sin hijos          6.572 2446
Hombres   Q3 (40-45h)   Con hijos          6.448  409
Hombres   Q3 (40-45h)   Sin hijos          6.595  721
Hombres  Q4 (45-168h)   Con hijos          6.482  824
Hombres  Q4 (45-168h)   Sin hijos          6.535 1083
Mujeres    Q1 (0-25h)   Con hijos          6.204 1400
Mujeres    Q1 (0-25h)   Sin hijos          6.379 1048
Mujeres   Q2 (25-40h)   Con hijos          6.361 1843
Mujeres   Q2 (25-40h)   Sin hijos          6.515 2346
Mujeres   Q3 (40-45h)   Con hijos          6.323  369
Mujeres   Q3 (40-45h)   Sin hijos          6.581  495
Mujeres  Q4 (45-168h)   Con hijos          6.243  539
Mujeres  Q4 (45-168h)   Sin hijos          6.420  570


## 5. Exploración: tablas pivot por género

In [8]:
for genero in ['Hombres', 'Mujeres']:
    print(f'\n=== {genero}: IDX_WELLBEING medio ===')
    sub = hm[hm['genero'] == genero].pivot(
        index='quintil_horas', columns='grupo_hijos', values='idx_wellbeing'
    )
    cols_order = [c for c in ['Sin hijos', 'Con hijos'] if c in sub.columns]
    print(sub[cols_order].to_string())


=== Hombres: IDX_WELLBEING medio ===
grupo_hijos    Sin hijos  Con hijos
quintil_horas                      
Q1 (0-25h)         6.469      6.326
Q2 (25-40h)        6.572      6.474
Q3 (40-45h)        6.595      6.448
Q4 (45-168h)       6.535      6.482

=== Mujeres: IDX_WELLBEING medio ===
grupo_hijos    Sin hijos  Con hijos
quintil_horas                      
Q1 (0-25h)         6.379      6.204
Q2 (25-40h)        6.515      6.361
Q3 (40-45h)        6.581      6.323
Q4 (45-168h)       6.420      6.243


In [9]:
print('=== Diferencia Hombres − Mujeres por celda ===')
hm_h = hm[hm['genero']=='Hombres'][['quintil_horas','grupo_hijos','idx_wellbeing']].rename(columns={'idx_wellbeing':'wb_h'})
hm_m = hm[hm['genero']=='Mujeres'][['quintil_horas','grupo_hijos','idx_wellbeing']].rename(columns={'idx_wellbeing':'wb_m'})
diff = hm_h.merge(hm_m, on=['quintil_horas','grupo_hijos'])
diff['H_minus_M'] = (diff['wb_h'] - diff['wb_m']).round(3)

pivot_diff = diff.pivot(index='quintil_horas', columns='grupo_hijos', values='H_minus_M')
cols_order = [c for c in ['Sin hijos', 'Con hijos'] if c in pivot_diff.columns]
print(pivot_diff[cols_order].to_string())
print()

idx_max = diff['H_minus_M'].idxmax()
print('Celda con mayor penalización para mujeres (H−M más alto):')
print(diff.loc[idx_max][['quintil_horas','grupo_hijos','wb_h','wb_m','H_minus_M']].to_string())

=== Diferencia Hombres − Mujeres por celda ===
grupo_hijos    Sin hijos  Con hijos
quintil_horas                      
Q1 (0-25h)         0.090      0.122
Q2 (25-40h)        0.057      0.113
Q3 (40-45h)        0.014      0.125
Q4 (45-168h)       0.115      0.239

Celda con mayor penalización para mujeres (H−M más alto):
quintil_horas    Q4 (45-168h)
grupo_hijos         Con hijos
wb_h                    6.482
wb_m                    6.243
H_minus_M               0.239


In [10]:
print('=== N por celda (Hombres) ===')
cols_order = [c for c in ['Sin hijos', 'Con hijos']]
print(hm[hm['genero']=='Hombres'].pivot(index='quintil_horas', columns='grupo_hijos', values='n').to_string())

print('\n=== N por celda (Mujeres) ===')
print(hm[hm['genero']=='Mujeres'].pivot(index='quintil_horas', columns='grupo_hijos', values='n').to_string())

min_row = hm.loc[hm['n'].idxmin()]
print(f'\nCelda con menos datos: {min_row["genero"]} | {min_row["quintil_horas"]} | {min_row["grupo_hijos"]} → n={min_row["n"]}')

=== N por celda (Hombres) ===
grupo_hijos    Con hijos  Sin hijos
quintil_horas                      
Q1 (0-25h)           472        722
Q2 (25-40h)         1302       2446
Q3 (40-45h)          409        721
Q4 (45-168h)         824       1083

=== N por celda (Mujeres) ===
grupo_hijos    Con hijos  Sin hijos
quintil_horas                      
Q1 (0-25h)          1400       1048
Q2 (25-40h)         1843       2346
Q3 (40-45h)          369        495
Q4 (45-168h)         539        570

Celda con menos datos: Mujeres | Q3 (40-45h) | Con hijos → n=369


## 6. Verificar y guardar

In [11]:
def verificar_csv_flourish(df, nombre):
    print(f'\n=== {nombre} ===')
    print(f'Filas: {len(df)}')
    print(f'Columnas: {list(df.columns)}')
    nulos = df.isnull().sum()
    nulos_con = nulos[nulos > 0]
    if len(nulos_con):
        print(f'Valores nulos:\n{nulos_con}')
    else:
        print('Sin valores nulos')
    assert not df.isin([float('inf'), float('-inf')]).any().any(), 'Hay infinitos'
    assert not df.isnull().all(axis=1).any(), 'Hay filas completamente nulas'
    print(f'Muestra:\n{df.head(4).to_string()}')
    print('✓ OK')

verificar_csv_flourish(hm, 'v3_heatmap_empleo')

out_path = OUT / 'v3_heatmap_empleo.csv'
hm.to_csv(out_path, index=False)
print(f'\nGuardado: {out_path}')


=== v3_heatmap_empleo ===
Filas: 16
Columnas: ['genero', 'quintil_horas', 'grupo_hijos', 'idx_wellbeing', 'n']
Sin valores nulos
Muestra:
    genero quintil_horas grupo_hijos  idx_wellbeing     n
0  Hombres    Q1 (0-25h)   Con hijos          6.326   472
1  Hombres    Q1 (0-25h)   Sin hijos          6.469   722
2  Hombres   Q2 (25-40h)   Con hijos          6.474  1302
3  Hombres   Q2 (25-40h)   Sin hijos          6.572  2446
✓ OK

Guardado: ../data/processed/v3_heatmap_empleo.csv


## 7. Resumen

In [12]:
print('=' * 60)
print('RESUMEN V3 — HEATMAP EMPLEO')
print('=' * 60)
print()
print(f'Población base: {len(activos_hm):,} activos con IDX_WELLBEING y chldhhe válido')
print(f'Celdas heatmap: {len(hm)}')
print(f'Rango IDX_WELLBEING: [{hm["idx_wellbeing"].min():.3f}, {hm["idx_wellbeing"].max():.3f}]')
print()

worst = hm.loc[hm['idx_wellbeing'].idxmin()]
best  = hm.loc[hm['idx_wellbeing'].idxmax()]
print(f'Peor bienestar:  {worst.genero} | {worst.quintil_horas} | {worst.grupo_hijos} → {worst.idx_wellbeing}')
print(f'Mejor bienestar: {best.genero} | {best.quintil_horas} | {best.grupo_hijos} → {best.idx_wellbeing}')
print()
print('Nota: chldhhe en ESS R11 es binaria (1=con hijos, 2=sin hijos en el hogar)')
print()
print('Configuración Flourish:')
print('  Tipo: Heatmap')
print('  Filas: quintil_horas')
print('  Columnas: grupo_hijos (Con hijos / Sin hijos)')
print('  Valor de color: idx_wellbeing')
print('  Panel/filtro: genero → dos heatmaps en paralelo')
print('  Paleta: verde (alto bienestar) → rojo (bajo bienestar)')

RESUMEN V3 — HEATMAP EMPLEO

Población base: 16,589 activos con IDX_WELLBEING y chldhhe válido
Celdas heatmap: 16
Rango IDX_WELLBEING: [6.204, 6.595]

Peor bienestar:  Mujeres | Q1 (0-25h) | Con hijos → 6.204
Mejor bienestar: Hombres | Q3 (40-45h) | Sin hijos → 6.595

Nota: chldhhe en ESS R11 es binaria (1=con hijos, 2=sin hijos en el hogar)

Configuración Flourish:
  Tipo: Heatmap
  Filas: quintil_horas
  Columnas: grupo_hijos (Con hijos / Sin hijos)
  Valor de color: idx_wellbeing
  Panel/filtro: genero → dos heatmaps en paralelo
  Paleta: verde (alto bienestar) → rojo (bajo bienestar)
